# Laboratorio de preparación para el examen final

Este cuaderno verifica conexiones entre teoría espectral, mínimos cuadrados, SVD, pseudoinversa y aproximación de rango bajo. Antes de ejecutar cada celda, anticipa el resultado y realiza los cálculos esenciales por escrito.

## 1. Herramientas y convenciones

Usaremos SymPy para resultados exactos y NumPy para cálculos numéricos. Las comprobaciones con `assert` forman parte del laboratorio: una celda que termina sin error confirma las identidades indicadas.

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()

## 2. Teoría espectral, forma cuadrática y valores singulares

Para la matriz simétrica

$$C=\begin{pmatrix}2&-1&0\\-1&2&0\\0&0&-3\end{pmatrix},$$

calcula primero sus valores propios, clasifica $x^TCx$ y predice sus valores singulares.

In [ ]:
C = sp.Matrix([[2, -1, 0], [-1, 2, 0], [0, 0, -3]])
espectro_C = C.eigenvects()
singulares_C = C.singular_values()
assert sorted(singulares_C, reverse=True) == [3, 3, 1]
espectro_C, singulares_C

La forma es indefinida porque el espectro contiene signos opuestos. Los valores singulares conservan las magnitudes, no los signos. Construye a mano una SVD a partir de la diagonalización espectral y explica qué columna de $U$ debe cambiar de signo.

## 3. SVD rectangular completa

Trabajamos ahora con la matriz del Ejercicio 7 de la guía. Antes de ejecutar, calcula $A^TA$ y anticipa el rango y las dimensiones de una SVD reducida.

In [ ]:
A = sp.Matrix([[1, -1], [0, 1], [1, 0]])
ATA = A.T * A
datos_espectrales = ATA.eigenvects()
assert A.rank() == 2
ATA, datos_espectrales

In [ ]:
# SymPy devuelve U y V con tantas columnas como el rango: SVD reducida.
U, S, V = A.singular_value_decomposition()
assert U.T * U == sp.eye(2)
assert V.T * V == sp.eye(2)
assert sp.simplify(U * S * V.T - A) == sp.zeros(3, 2)
U, S, V

**Actividad.** Halla un vector unitario de $\ker(A^T)$, añádelo como tercera columna de $U$ y construye la matriz $\Sigma$ de tamaño $3\times2$. Verifica que obtienes una SVD completa.

## 4. Pseudoinversa, proyecciones y mínimos cuadrados

Usaremos un ajuste lineal con columnas independientes. Compara la solución por ecuaciones normales con la obtenida mediante la pseudoinversa.

In [ ]:
M = sp.Matrix([[1, 0], [1, 1], [1, 2]])
b = sp.Matrix([1, 2, 2])
x_normal = (M.T * M).inv() * M.T * b
M_plus = M.pinv()
x_plus = M_plus * b
ajuste = M * x_plus
residuo = b - ajuste
assert x_normal == x_plus == sp.Matrix([sp.Rational(7, 6), sp.Rational(1, 2)])
assert M.T * residuo == sp.zeros(2, 1)
x_plus, ajuste, residuo

In [ ]:
P_col = M * M_plus
P_fila = M_plus * M
assert P_col.T == P_col and P_col * P_col == P_col
assert P_fila == sp.eye(2)
assert M * M_plus * M == M
assert M_plus * M * M_plus == M_plus
P_col, P_fila

## 5. Norma mínima cuando hay infinitas soluciones

Para $Dx=b$, describe primero la familia de soluciones. Después verifica que la pseudoinversa selecciona la única solución ortogonal al núcleo.

In [ ]:
D = sp.Matrix([[1, 1, 0], [0, 1, 1]])
d = sp.Matrix([2, 1])
x_min = D.pinv() * d
nucleo = D.nullspace()[0]
assert D * x_min == d
assert (x_min.T * nucleo)[0] == 0
assert x_min == sp.Matrix([1, 1, 0])
x_min, nucleo

## 6. Aproximación de rango bajo

Construiremos una matriz con valores singulares conocidos $10,4,2,1$. Verifica las fórmulas de error y determina el menor rango que captura al menos el $95\%$ de la energía.

In [ ]:
theta = np.pi / 7
Q1 = np.array([[np.cos(theta), -np.sin(theta), 0, 0],
               [np.sin(theta),  np.cos(theta), 0, 0],
               [0, 0, 1, 0], [0, 0, 0, 1]])
phi = np.pi / 5
Q2 = np.array([[1, 0, 0, 0], [0, np.cos(phi), -np.sin(phi), 0],
               [0, np.sin(phi),  np.cos(phi), 0], [0, 0, 0, 1]])
X = Q1 @ np.diag([10., 4., 2., 1.]) @ Q2.T
Ux, sx, Vtx = np.linalg.svd(X, full_matrices=False)
energia_acumulada = np.cumsum(sx**2) / np.sum(sx**2)
k95 = int(np.searchsorted(energia_acumulada, 0.95) + 1)
assert np.allclose(sx, [10, 4, 2, 1])
assert k95 == 2
sx, energia_acumulada, k95

In [ ]:
k = 2
X2 = (Ux[:, :k] * sx[:k]) @ Vtx[:k, :]
error_2 = np.linalg.norm(X - X2, 2)
error_F = np.linalg.norm(X - X2, 'fro')
assert np.isclose(error_2, 2)
assert np.isclose(error_F, np.sqrt(5))
error_2, error_F

## 7. Aplicación visual: truncamiento de una imagen

Retomamos la imagen usada en C22. La celda usa la copia local cuando está disponible y, en Google Colab, la descarga automáticamente desde el repositorio público. Compara varios rangos y relaciona calidad, energía y almacenamiento.

In [ ]:
from pathlib import Path
from PIL import Image

candidatos = [Path('unidad_4/figuras/ojo_svd.jpg'), Path('figuras/ojo_svd.jpg')]
ruta = next((p for p in candidatos if p.exists()), None)
if ruta is None:
    from urllib.request import urlretrieve
    ruta = Path('ojo_svd.jpg')
    url_imagen = ('https://raw.githubusercontent.com/ovelasquezc/'
                  'algebra-lineal-aplicada/main/unidad_4/figuras/ojo_svd.jpg')
    urlretrieve(url_imagen, ruta)
imagen = np.asarray(Image.open(ruta).convert('L'), dtype=float) / 255
Ui, si, Vti = np.linalg.svd(imagen, full_matrices=False)
rangos = [5, 15, 40]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(imagen, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original')
for ax, k in zip(axes[1:], rangos):
    Ak = (Ui[:, :k] * si[:k]) @ Vti[:k, :]
    energia = np.sum(si[:k]**2) / np.sum(si**2)
    ax.imshow(Ak, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'rango {k}\nenergía {energia:.1%}')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Cierre

Sin consultar las celdas anteriores, explica por escrito: (1) por qué $A^+b$ conecta proyección y norma mínima; (2) qué información del espectro se pierde al pasar a valores singulares; y (3) por qué la SVD truncada es óptima, no solo una compresión posible.